In [1]:
from ucimlrepo import fetch_ucirepo 
import pandas as pd 
# fetch dataset 
online_retail = fetch_ucirepo(id=352) 

  
# metadata 
print(online_retail.metadata) 
  
# variable information 
print(online_retail.variables) 
df = online_retail.data.original

print(df.info())
display(df.head())


{'uci_id': 352, 'name': 'Online Retail', 'repository_url': 'https://archive.ics.uci.edu/dataset/352/online+retail', 'data_url': 'https://archive.ics.uci.edu/static/public/352/data.csv', 'abstract': 'This is a transactional data set which contains all the transactions occurring between 01/12/2010 and 09/12/2011 for a UK-based and registered non-store online retail.', 'area': 'Business', 'tasks': ['Classification', 'Clustering'], 'characteristics': ['Multivariate', 'Sequential', 'Time-Series'], 'num_instances': 541909, 'num_features': 6, 'feature_types': ['Integer', 'Real'], 'demographics': [], 'target_col': None, 'index_col': ['InvoiceNo', 'StockCode'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2015, 'last_updated': 'Mon Oct 21 2024', 'dataset_doi': '10.24432/C5BW33', 'creators': ['Daqing Chen'], 'intro_paper': {'ID': 361, 'type': 'NATIVE', 'title': 'Data mining for the online retail industry: A case study of RFM model-based customer segmenta

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


In [3]:
# 1. Drop rows where CustomerID is missing
df_clean = df.dropna(subset=['CustomerID']).copy()

# 2. Filter out canceled orders (Quantity < 0 or InvoiceNo starts with 'C')
df_clean = df_clean[df_clean['Quantity'] > 0]

# 3. Convert InvoiceDate to proper datetime objects
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# 4. Calculate the Total Spend for each line item (Quantity * UnitPrice)
df_clean['Total_spend']  = df_clean['Quantity'] * df_clean['UnitPrice']

print(f"original rows{len(df)}")

print(f"cleaned rows{len(df_clean)}")

original rows541909
cleaned rows397924


In [5]:
# 1. Create a "Snapshot Date" (1 day after the most recent purchase in the entire dataset)
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

# 2. Aggregate the data at the Customer level
rfm = df_clean.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (snapshot_date - x.max()).days,  # Recency: Days since last purchase
    'InvoiceNo': 'nunique',                                   # Frequency: Total unique invoices
    'Total_spend': 'sum'                                       # Monetary: Total money spent
})

# 3. Rename columns so it looks professional
rfm.rename(columns={
    'InvoiceDate': 'Recency',
    'InvoiceNo': 'Frequency',
    'Total_spend': 'Monetary'
}, inplace=True)

# 4. Check the stats
print(rfm.describe())
display(rfm.head())

           Recency    Frequency       Monetary
count  4339.000000  4339.000000    4339.000000
mean     92.518322     4.271952    2053.793018
std     100.009747     7.705493    8988.248381
min       1.000000     1.000000       0.000000
25%      18.000000     1.000000     307.245000
50%      51.000000     2.000000     674.450000
75%     142.000000     5.000000    1661.640000
max     374.000000   210.000000  280206.020000


,Recency,Frequency,Monetary
CustomerID,,,
12346.0,326,1,77183.60
12347.0,2,7,4310.00
12348.0,75,4,1797.24
12349.0,19,1,1757.55
12350.0,310,1,334.40


In [6]:
# 1. Create labels for our scores
r_labels = range(5, 0, -1) # [5, 4, 3, 2, 1] (Lower recency = higher score)
f_labels = range(1, 6)     # [1, 2, 3, 4, 5] (Higher frequency = higher score)
m_labels = range(1, 6)     # [1, 2, 3, 4, 5] (Higher spend = higher score)

# 2. Assign scores to buckets
rfm['R_Score'] = pd.qcut(rfm['Recency'], q=5, labels=r_labels)

# We use .rank(method='first') on Frequency because lots of people only bought 1 time.
# Without rank, qcut crashes because it can't figure out how to split ties.
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), q=5, labels=f_labels)

rfm['M_Score'] = pd.qcut(rfm['Monetary'], q=5, labels=m_labels)

# 3. Combine them into a master RFM score (e.g., '555' is a perfect customer)
rfm['RFM_Class'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

display(rfm.head())

,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Class
CustomerID,,,,,,,
12346.0,326,1,77183.60,1,1,5,115
12347.0,2,7,4310.00,5,5,5,555
12348.0,75,4,1797.24,2,4,4,244
12349.0,19,1,1757.55,4,1,4,414
12350.0,310,1,334.40,1,1,2,112


In [7]:
# Convert scores to integers so we can evaluate them easily
rfm['R_Score'] = rfm['R_Score'].astype(int)
rfm['F_Score'] = rfm['F_Score'].astype(int)

# Define the segmentation logic
def assign_segment(row):
    r, f = row['R_Score'], row['F_Score']
    
    if r >= 4 and f >= 4:
        return 'Champions'
    elif r >= 3 and f >= 3:
        return 'Loyal Customers'
    elif r >= 4 and f <= 2:
        return 'New/Promising'
    elif r == 3 and f <= 2:
        return 'Needs Attention'
    elif r <= 2 and f >= 3:
        return 'At Risk / Churning'
    else:
        return 'Lost'

# Apply the function to create our final Segment column
rfm['Customer_Segment'] = rfm.apply(assign_segment, axis=1)

# See the total count of customers in each segment
segment_counts = rfm['Customer_Segment'].value_counts()
print(segment_counts)

Customer_Segment
Champions             1139
Lost                  1065
Loyal Customers        821
At Risk / Churning     643
Needs Attention        351
New/Promising          320
Name: count, dtype: int64


In [8]:
# 1. Aggregate revenue and customer count per segment
segment_revenue = rfm.groupby('Customer_Segment').agg({
    'Monetary': 'sum',
    'Recency': 'count' # We just use this to count the number of customers
}).rename(columns={'Recency': 'Customer_Count', 'Monetary': 'Total_Revenue'})

# 2. Calculate what percentage of total money comes from each segment
total_money = segment_revenue['Total_Revenue'].sum()
segment_revenue['Revenue_Share_%'] = (segment_revenue['Total_Revenue'] / total_money) * 100

# 3. Sort by highest revenue
segment_revenue = segment_revenue.sort_values(by='Total_Revenue', ascending=False)

# Format the output so it looks like currency, not messy scientific notation
pd.options.display.float_format = '{:,.2f}'.format
display(segment_revenue)

# 4. EXPORT THE FINAL DATASET
rfm.to_csv("rfm_customer_segments.csv")
print("✅ Saved to rfm_customer_segments.csv!")

,Total_Revenue,Customer_Count,Revenue_Share_%
Customer_Segment,,,
Champions,"5,927,723.91",1139,66.52
Loyal Customers,"1,355,744.71",821,15.21
At Risk / Churning,"800,531.55",643,8.98
Lost,"519,408.57",1065,5.83
Needs Attention,"161,832.59",351,1.82
New/Promising,"146,166.57",320,1.64


✅ Saved to rfm_customer_segments.csv!
